In [1]:
# Install all required packages before importing notebook dependencies.
import importlib
import os
import subprocess
import sys
from pathlib import Path

PIP_PACKAGES = [
    "kaggle",
    "pandas",
    "python-dotenv",
    "pyarrow",
    "fastparquet",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    *PIP_PACKAGES,
])

def progress(message):
    print(f"[progress] {message}")

REQUIRED_PACKAGES = {
    "kaggle": "kaggle",
    "pandas": "pandas",
    "dotenv": "python-dotenv",
    "pyarrow": "pyarrow",
    "fastparquet": "fastparquet",
}

for import_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import pandas as pd
from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

current_dir = Path.cwd().resolve()
env_candidates = [current_dir / ".env", current_dir / "training" / ".env"]
if current_dir.name.lower() == "training":
    env_candidates.insert(0, current_dir.parent / ".env")

for env_path in env_candidates:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"Loaded environment from {env_path}")
        break

progress("Setup complete: required libraries are installed and imported.")
print("Required libraries are installed and imported.")

Loaded environment from C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\.env
[progress] Setup complete: required libraries are installed and imported.
Required libraries are installed and imported.


# Download CIC-DDoS2019 Dataset

This notebook downloads the CIC-DDoS2019 DDoS dataset into `training/dataset` for ADS-Detect model training.

The official CIC page uses a web form, so this notebook uses a Kaggle mirror by default. Before running the download cell, make sure you have Kaggle API credentials configured:

1. Create or sign in to a Kaggle account.
2. Go to `Account > API > Create New Token`.
3. Put the downloaded `kaggle.json` file in `C:\Users\<your-user>\.kaggle\kaggle.json`.

Alternative: fill in `KAGGLE_USERNAME` and `KAGGLE_KEY` inside the project root `.env` file, or set them as environment variables.

In [2]:
cwd = Path.cwd().resolve()
training_dir = cwd if cwd.name.lower() == "training" else cwd / "training"
dataset_dir = training_dir / "dataset"
dataset_dir.mkdir(parents=True, exist_ok=True)

progress("Preparing dataset directory...")
print(f"Notebook running from: {cwd}")
print(f"Dataset will be stored in: {dataset_dir}")
progress("Dataset directory is ready.")

[progress] Preparing dataset directory...
Notebook running from: C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\training
Dataset will be stored in: C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\training\dataset
[progress] Dataset directory is ready.


## Choose Dataset Mirror

Default mirror:
- `dhoogla/cicddos2019` (this mirror currently extracts `.parquet` files)

Larger full CSV mirror option:
- `rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files`

If the default mirror does not include the files you need, replace `DATASET_SLUG` below with the larger mirror.

In [3]:
DATASET_SLUG = "dhoogla/cicddos2019"

# Uncomment this instead if you need the larger full CSV mirror.
# DATASET_SLUG = "rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files"

progress(f"Dataset mirror selected: {DATASET_SLUG}")

[progress] Dataset mirror selected: dhoogla/cicddos2019


In [4]:
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
has_env_credentials = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))

if not kaggle_json.exists() and not has_env_credentials:
    raise FileNotFoundError(
        "Kaggle credentials were not found. Put kaggle.json in "
        f"{kaggle_json} or set KAGGLE_USERNAME and KAGGLE_KEY."
    )

progress("Kaggle credentials check passed.")
print("Kaggle credentials found.")

[progress] Kaggle credentials check passed.
Kaggle credentials found.


In [5]:
api = KaggleApi()
api.authenticate()

progress(f"Starting download for {DATASET_SLUG}...")
print(f"Downloading {DATASET_SLUG}...")
api.dataset_download_files(
    DATASET_SLUG,
    path=str(dataset_dir),
    unzip=True,
    quiet=False,
)
progress("Download finished and files were extracted.")
print("Download complete.")

[progress] Starting download for dhoogla/cicddos2019...
Dataset URL: https://www.kaggle.com/datasets/dhoogla/cicddos2019
... resuming from 2097152 bytes (28044083 bytes left) ...


100%|██████████| 28.7M/28.7M [05:37<00:00, 83.1kB/s]



[progress] Download finished and files were extracted.
Download complete.


In [6]:
csv_files = sorted(dataset_dir.rglob("*.csv"))
parquet_files = sorted(dataset_dir.rglob("*.parquet"))
pcap_files = sorted(dataset_dir.rglob("*.pcap")) + sorted(dataset_dir.rglob("*.pcapng"))

progress("Scanning downloaded files...")
print(f"CSV files found: {len(csv_files)}")
print(f"Parquet files found: {len(parquet_files)}")
print(f"PCAP files found: {len(pcap_files)}")

for file_path in csv_files[:20]:
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"{file_path.relative_to(dataset_dir)} - {size_mb:.2f} MB")

if len(csv_files) > 20:
    print(f"... and {len(csv_files) - 20} more CSV files")

for file_path in parquet_files[:20]:
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"{file_path.relative_to(dataset_dir)} - {size_mb:.2f} MB")
if len(parquet_files) > 20:
    print(f"... and {len(parquet_files) - 20} more Parquet files")

progress("File scan complete.")

[progress] Scanning downloaded files...
CSV files found: 0
Parquet files found: 17
PCAP files found: 0
DNS-testing.parquet - 0.49 MB
LDAP-testing.parquet - 0.21 MB
LDAP-training.parquet - 0.59 MB
MSSQL-testing.parquet - 0.40 MB
MSSQL-training.parquet - 0.55 MB
NetBIOS-testing.parquet - 0.24 MB
NetBIOS-training.parquet - 0.21 MB
NTP-testing.parquet - 9.05 MB
Portmap-training.parquet - 0.54 MB
SNMP-testing.parquet - 0.26 MB
Syn-testing.parquet - 0.13 MB
Syn-training.parquet - 8.21 MB
TFTP-testing.parquet - 7.86 MB
UDP-testing.parquet - 0.95 MB
UDP-training.parquet - 1.32 MB
UDPLag-testing.parquet - 1.21 MB
UDPLag-training.parquet - 1.36 MB
[progress] File scan complete.


In [7]:
sample_file = None
sample_loader = None

if csv_files:
    sample_file = csv_files[0]
    sample_loader = lambda path: pd.read_csv(path, nrows=5)
elif parquet_files:
    sample_file = parquet_files[0]
    sample_loader = lambda path: pd.read_parquet(path).head(5)

if sample_file is not None:
    progress(f"Previewing the first tabular file: {sample_file.name}...")
    try:
        sample_df = sample_loader(sample_file)
        print(f"Sample file: {sample_file.relative_to(dataset_dir)}")
        print(f"Columns: {len(sample_df.columns)}")
        display(sample_df)
        progress("Sample preview complete.")
    except ImportError:
        print("Parquet files were found, but no parquet engine is installed in this environment.")
        print("Install `pyarrow` or `fastparquet`, then rerun this cell if you want a data preview.")
        progress("Sample preview skipped because parquet support is unavailable.")
else:
    print("No CSV or Parquet files were found. Check the downloaded files or try a different mirror slug.")
    progress("Sample preview skipped because no tabular files were found.")

[progress] Previewing the first tabular file: DNS-testing.parquet...
Sample file: DNS-testing.parquet
Columns: 78


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,17,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
1,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
3,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
4,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS


[progress] Sample preview complete.


## Convert Parquet Files to CSV

Run this after the file scan if you need CSV copies for training code that does not read Parquet directly.

In [8]:
csv_output_dir = dataset_dir / "csv"
csv_output_dir.mkdir(parents=True, exist_ok=True)

if not parquet_files:
    print("No Parquet files were found, so there is nothing to convert.")
    progress("CSV conversion skipped because no Parquet files were found.")
else:
    try:
        # pandas needs pyarrow or fastparquet to read Parquet files.
        pd.read_parquet(parquet_files[0]).head(1)
    except ImportError:
        print("Parquet files were found, but no Parquet engine is installed.")
        print("Install one of these in the active notebook environment, then rerun this cell:")
        print(f"{sys.executable} -m pip install pyarrow")
        print("or use Anaconda Navigator / Anaconda Prompt to install pyarrow into ejie_dev3.10.")
        progress("CSV conversion skipped because Parquet support is unavailable.")
    else:
        progress(f"Converting {len(parquet_files)} Parquet files to CSV...")
        converted_files = []

        for parquet_path in parquet_files:
            csv_path = csv_output_dir / f"{parquet_path.stem}.csv"
            df = pd.read_parquet(parquet_path)
            df.to_csv(csv_path, index=False)
            converted_files.append(csv_path)
            size_mb = csv_path.stat().st_size / (1024 * 1024)
            print(f"Created {csv_path.relative_to(dataset_dir)} - {size_mb:.2f} MB")

        csv_files = sorted(dataset_dir.rglob("*.csv"))
        print(f"CSV files now available: {len(csv_files)}")
        progress("CSV conversion complete.")

[progress] Converting 17 Parquet files to CSV...
Created csv\DNS-testing.csv - 2.13 MB
Created csv\LDAP-testing.csv - 0.90 MB
Created csv\LDAP-training.csv - 2.19 MB
Created csv\MSSQL-testing.csv - 2.48 MB
Created csv\MSSQL-training.csv - 3.33 MB
Created csv\NetBIOS-testing.csv - 0.74 MB
Created csv\NetBIOS-training.csv - 0.56 MB
Created csv\NTP-testing.csv - 46.57 MB
Created csv\Portmap-training.csv - 1.73 MB
Created csv\SNMP-testing.csv - 1.25 MB
Created csv\Syn-testing.csv - 0.30 MB
Created csv\Syn-training.csv - 25.56 MB
Created csv\TFTP-testing.csv - 41.69 MB
Created csv\UDP-testing.csv - 4.37 MB
Created csv\UDP-training.csv - 6.15 MB
Created csv\UDPLag-testing.csv - 4.24 MB
Created csv\UDPLag-training.csv - 4.42 MB
CSV files now available: 17
[progress] CSV conversion complete.
